# Outage model playground

A template for experimenting with outage prediction. **Change the `CONFIG` cell, then Run All.**
Each run builds features, trains a model, scores it against a simple baseline, and logs the result to `experiments.csv`, so you can compare runs.

```
stg.merged_hourly ──► features (SQL window functions) ──► target ──► split by time ──► model ──► scores + plots ──► experiments.csv
   (pixi run build)         Step 1                          Step 2      Step 3           Step 4      Steps 5–7
```

**The prediction moment.** Every row is "a region at the start of hour *t*". At that moment we know:
- the weather report from just before *t* (the :51 report — that's why the SQL pipeline shifted it +9 min)
- outages up to hour *t − 1*

and we predict what happens over the next `horizon_h` hours, *t* … *t + horizon − 1*. Every feature must respect that, or the model is cheating (**leakage**) and its scores are fake.

**Requirements:** use the pixi kernel (`.pixi/envs/default/bin/python`) after `pixi run build`. This notebook opens `data/warehouse.duckdb` read-only, so close anything that has it open for writing (DuckDB UI, a running build) first — and restart this kernel before running `pixi run build` again.

In [ ]:
CONFIG = dict(
    # --- What to predict -----------------------------------------------------
    target_col = "pct_out",     # "pct_out" (% of customers out) or "peak_out" (customer count)
    task       = "classify",    # "classify": will it exceed `threshold`?   "regress": how big will it get?
    threshold  = 1.0,           # classify only: event = max(target_col over the horizon) > threshold
    horizon_h  = 6,             # how many hours ahead to look (1 = just this hour)

    # --- What the model is allowed to see ------------------------------------
    weather_now = ["temp_c", "wind_ms", "gust_ms", "precip_mm", "thunder"],   # the latest report
    rolling     = {"gust_ms": "max", "precip_mm": "sum", "thunder": "max"},   # column -> how to summarize the past window
    windows_h   = [3, 6, 24],   # rolling windows: the last N hours of weather
    lags_h      = [1, 3, 24],   # past outage levels: target_col N hours ago (1 = last hour)
    use_region  = True,         # one column per region (lets the model learn each region's habits)
    use_calendar = True,        # hour of day (UTC), month, day of week
    perfect_forecast = False,   # True = ALSO show the ACTUAL weather of the coming horizon.
                                #   That's cheating on purpose: it acts like a perfect forecast and
                                #   tells you the best you could do with weather forecasts.

    # --- Train / test split: by time, never random ---------------------------
    train_years = (2020, 2022), # inclusive. Before 2020 the data has coverage gaps (see merge_all.sql)
    test_years  = (2023, 2025),

    # --- Model ---------------------------------------------------------------
    model = "hgb",              # "logistic", "tree", or "hgb" (gradient boosting)
    decision_threshold = 0.5,   # classify only: predicted probability above this = "alarm"
    notes = "",                 # free text, saved with the run in experiments.csv
)

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn import metrics

con = duckdb.connect("../data/warehouse.duckdb", read_only=True)
con.sql("SELECT county, count(*) AS hours, min(hour) AS first, max(hour) AS last FROM stg.merged_hourly GROUP BY county ORDER BY county").df()

## Step 1 — Features (SQL window functions)

A **window function** computes something over *neighbouring rows* without squashing them like `GROUP BY` does. Each region is its own timeline (`PARTITION BY fips_code ORDER BY hour`), and the *frame* says which neighbours:

| SQL | Meaning at hour *t* | Used for |
|---|---|---|
| `lag(x, 3)` | x at *t − 3* | past outages |
| `max(x) ... ROWS BETWEEN 5 PRECEDING AND CURRENT ROW` | max over *t − 5 … t* | past weather |
| `max(x) ... ROWS BETWEEN CURRENT ROW AND 5 FOLLOWING` | max over *t … t + 5* | **the target** (the future!) |

Counting rows = counting hours only works because the grid has **every** hour — that's what the CROSS JOIN in `merge_all.sql` bought us.

The function below writes the SQL from `CONFIG`; the next cell prints it so you can read (and copy) it. Two leakage rules are built in: past-outage features start at lag 1 (hour *t* isn't over yet), and only `perfect_forecast` may look at `FOLLOWING` rows.

In [ ]:
def build_feature_sql(cfg):
    t = cfg["target_col"]
    H = cfg["horizon_h"]

    def over(frame=""):
        return f"OVER (PARTITION BY fips_code ORDER BY hour {frame})"

    cols = [
        "fips_code", "county", "hour", "customers",
        # the TARGET: worst value over the next H hours, starting now
        f"max({t}) {over(f'ROWS BETWEEN CURRENT ROW AND {H - 1} FOLLOWING')} AS target_value",
        # the BASELINE ('persistence'): assume the next H hours look like the last hour
        f"lag({t}, 1) {over()} AS _persistence",
    ]
    if cfg["use_calendar"]:
        cols += ["hour(hour) AS hour_utc", "month(hour) AS month", "dayofweek(hour) AS dow"]

    cols += [f"{c} AS {c}_now" for c in cfg["weather_now"]]

    for c, agg in cfg["rolling"].items():
        for k in cfg["windows_h"]:
            cols.append(f"{agg}({c}) {over(f'ROWS BETWEEN {k - 1} PRECEDING AND CURRENT ROW')} AS {c}_{agg}_{k}h")

    for k in cfg["lags_h"]:
        cols.append(f"lag({t}, {k}) {over()} AS {t}_lag_{k}h")

    if cfg["perfect_forecast"]:
        for c, agg in cfg["rolling"].items():
            cols.append(f"{agg}({c}) {over(f'ROWS BETWEEN CURRENT ROW AND {H - 1} FOLLOWING')} AS future_{c}_{agg}")

    # thunder is a true/false column; CAST makes it 0/1 so max()/sum() work on it
    return ("WITH base AS (SELECT * REPLACE (CAST(thunder AS INT) AS thunder) FROM stg.merged_hourly)\n"
            "SELECT\n    " + ",\n    ".join(cols) + "\nFROM base\nORDER BY fips_code, hour")


sql = build_feature_sql(CONFIG)
print(sql)

In [ ]:
# Run it. Features are computed over ALL years first; we split afterwards.
# (A WHERE on years would run BEFORE the window functions and break the first lags.)
df = con.sql(sql).df()
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

## Step 2 — The target

`target_value` is the worst `target_col` over the coming horizon. For **classify** we turn it into yes/no with `threshold`; for **regress** we predict the number itself.

Rows with an unknown target (EAGLE-I down, blank snapshots, or the very end of the data) are dropped — we can't score a prediction against "unknown".

In [ ]:
df = df[df["target_value"].notna()].copy()
if CONFIG["task"] == "classify":
    df["y"] = (df["target_value"] > CONFIG["threshold"]).astype(int)
else:
    df["y"] = df["target_value"]

df["year"] = df["hour"].dt.year
if CONFIG["use_region"]:
    df = pd.concat([df, pd.get_dummies(df["county"], prefix="region", dtype=int)], axis=1)

# Everything that isn't an id, the answer, or the baseline is a feature
not_features = {"fips_code", "county", "hour", "year", "target_value", "y", "_persistence"}
FEATURES = [c for c in df.columns if c not in not_features]
print(len(FEATURES), "features:", FEATURES)

if CONFIG["task"] == "classify":
    print(df.groupby("county")["y"].mean().mul(100).round(1).rename("% of hours that are events"))

## Step 3 — Split by time

Train on earlier years, test on later ones, **never a random split**. Neighbouring hours are nearly identical, so a random split puts copies of test hours into training and the scores look far better than they'll be on the real future.

The test years are the "exam": peek at them for every run and you'll slowly tune the model to them. Strictly, you'd hold out a *validation* year for tuning and touch the test years once at the end — an exercise at the bottom.

In [ ]:
def in_years(frame, years):
    return frame["year"].between(*years)

train = df[in_years(df, CONFIG["train_years"])]
test  = df[in_years(df, CONFIG["test_years"])]
X_train, y_train = train[FEATURES], train["y"]
X_test,  y_test  = test[FEATURES],  test["y"]

summary = pd.DataFrame({
    "rows":  [len(train), len(test)],
    "from":  [train["hour"].min(), test["hour"].min()],
    "to":    [train["hour"].max(), test["hour"].max()],
}, index=["train", "test"])
if CONFIG["task"] == "classify":
    summary["% events"] = [100 * y_train.mean(), 100 * y_test.mean()]
summary

## Step 4 — Model

| name | what it is | NULLs |
|---|---|---|
| `logistic` | linear model: a weighted sum of features (needs scaling) | filled with the median |
| `tree` | one decision tree, small enough to read | handled natively |
| `hgb` | hundreds of small trees, each fixing the last one's mistakes — usually the strongest | handled natively |

`class_weight="balanced"` matters here: events are rare, and without it a model can score well by always saying "no".

In [ ]:
def make_model(name, task):
    classify = task == "classify"
    if name == "logistic":
        final = (LogisticRegression(max_iter=2000, class_weight="balanced") if classify else Ridge())
        return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), final)
    if name == "tree":
        return (DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=0) if classify
                else DecisionTreeRegressor(max_depth=4, random_state=0))
    if name == "hgb":
        return (HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=0)
                if classify else HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0))
    raise ValueError(f"unknown model: {name}")


model = make_model(CONFIG["model"], CONFIG["task"])
model.fit(X_train, y_train)

## Step 5 — Score it against a baseline

A score means nothing alone. The **persistence** baseline says "the next hours will look like the last hour". It's hard to beat for short horizons — outages last — so the real question is whether the model adds anything *beyond* "it's already out".

For classify:
- **precision** — of the alarms raised, how many were real events
- **recall** — of the real events, how many got an alarm
- **PR AUC** — precision vs recall over every possible `decision_threshold`; best single number for rare events (a coin flip scores ≈ the event rate)

In [ ]:
def score(task, y_true, y_pred, y_score=None):
    if task == "classify":
        out = {
            "precision": metrics.precision_score(y_true, y_pred, zero_division=0),
            "recall":    metrics.recall_score(y_true, y_pred, zero_division=0),
            "f1":        metrics.f1_score(y_true, y_pred, zero_division=0),
        }
        if y_score is not None:
            out["pr_auc"]  = metrics.average_precision_score(y_true, y_score)
            out["roc_auc"] = metrics.roc_auc_score(y_true, y_score)
        return out
    return {
        "mae":  metrics.mean_absolute_error(y_true, y_pred),
        "rmse": metrics.root_mean_squared_error(y_true, y_pred),
        "r2":   metrics.r2_score(y_true, y_pred),
    }


if CONFIG["task"] == "classify":
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= CONFIG["decision_threshold"]).astype(int)
    model_scores = score("classify", y_test, pred, proba)
    persist_pred = (test["_persistence"] > CONFIG["threshold"]).astype(int)   # NaN > x is False
    base_scores = score("classify", y_test, persist_pred)
else:
    pred = model.predict(X_test)
    model_scores = score("regress", y_test, pred)
    ok = test["_persistence"].notna()
    base_scores = score("regress", y_test[ok], test.loc[ok, "_persistence"])

results = pd.DataFrame({"model": model_scores, "persistence": base_scores}).round(3)
results

In [ ]:
if CONFIG["task"] == "classify":
    cm = pd.DataFrame(metrics.confusion_matrix(y_test, pred),
                      index=["actual: no event", "actual: event"],
                      columns=["predicted: no", "predicted: alarm"])
    display(cm)

## Step 6 — Look at one storm

Numbers hide *how* a model behaves. This plots the biggest event in the test period: did the alarm come before the outage, or only once it had already started (which persistence would also catch)?

In [ ]:
def plot_event(frame, pred_col, days=3):
    peak = frame.loc[frame["target_value"].idxmax()]
    win = frame[(frame["county"] == peak["county"]) &
                (frame["hour"].between(peak["hour"] - pd.Timedelta(days=days),
                                       peak["hour"] + pd.Timedelta(days=days)))]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax1.plot(win["hour"], win["target_value"], label=f"actual: max {CONFIG['target_col']} over next {CONFIG['horizon_h']}h")
    ax1.plot(win["hour"], win[f"{CONFIG['target_col']}_lag_1h"] if f"{CONFIG['target_col']}_lag_1h" in win else win["_persistence"],
             label="last hour (what persistence sees)", alpha=0.6)
    if CONFIG["task"] == "classify":
        ax1.axhline(CONFIG["threshold"], ls="--", c="gray", label="threshold")
    ax1.set_title(f"{peak['county']} around {peak['hour']:%Y-%m-%d %H:00} UTC")
    ax1.legend(loc="upper left")
    ax2.plot(win["hour"], win[pred_col], c="C3")
    ax2.set_ylabel("predicted probability" if CONFIG["task"] == "classify" else "predicted value")
    if CONFIG["task"] == "classify":
        ax2.axhline(CONFIG["decision_threshold"], ls="--", c="gray")
        ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


test = test.assign(prediction=proba if CONFIG["task"] == "classify" else pred)
plot_event(test, "prediction")

## Step 7 — What did the model rely on?

**Permutation importance**: shuffle one feature and see how much the score drops. A big drop means the model leaned on it. (Run on a sample of the test set, to keep it quick.)

In [ ]:
sample = test.sample(min(20_000, len(test)), random_state=0)
scoring = "average_precision" if CONFIG["task"] == "classify" else "neg_mean_absolute_error"
imp = permutation_importance(model, sample[FEATURES], sample["y"], scoring=scoring, n_repeats=3, random_state=0)
importance = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
importance.tail(15).plot.barh(figsize=(7, 5), title="Permutation importance (score drop when shuffled)")
plt.tight_layout()
plt.show()

## Step 8 — Log the run

Every run is appended to `experiments.csv`, with its config and scores, so you can compare runs side by side.

In [ ]:
log_path = Path("experiments.csv")
row = {"time": pd.Timestamp.now().floor("s"),
       **{k: (str(v) if isinstance(v, (list, tuple, dict)) else v) for k, v in CONFIG.items()},
       **{f"model_{k}": round(v, 4) for k, v in model_scores.items()},
       **{f"persist_{k}": round(v, 4) for k, v in base_scores.items()}}
log = pd.concat([pd.read_csv(log_path) if log_path.exists() else pd.DataFrame(), pd.DataFrame([row])], ignore_index=True)
log.to_csv(log_path, index=False)
log.tail(10)

## Things to try

One change per run, so you know what caused the difference. Write what you expect in `notes` *before* running.

1. **Horizon.** `horizon_h` = 1, 6, 24. Which falls faster as you look further ahead: the model, or persistence?
2. **How much is just "it's already out"?** Set `lags_h = []`. The model now sees only weather + calendar.
3. **The forecast ceiling.** `perfect_forecast = True`. The gap between this and step 2 is what good weather forecasts would be worth.
4. **Event size.** `threshold` = 0.5, 1, 5. Bigger events are rarer — what happens to precision?
5. **Model.** `logistic` vs `tree` vs `hgb`. Is the fancier model worth it?
6. **Coverage.** `train_years = (2015, 2019)`. The data then has coverage gaps (Jefferson County shows almost no outages in 2018–2019). Does the test score suffer?
7. **Alarm threshold.** Change `decision_threshold` (0.2, 0.8) and watch precision trade against recall. Which matters more for a utility?
8. **A new feature.** Add `"temp_c": "min"` to `rolling`, or write your own SQL column in `build_feature_sql` — e.g. an icing flag `CAST(temp_c <= 0 AND precip_mm > 0 AS INT) AS icing`.
9. **Regression.** `task = "regress"`. Then try predicting `ln(1 + pct_out)` instead — big storms dominate otherwise (hint: change the target line in `build_feature_sql`).
10. **Do it properly.** Train on 2020–2021, *tune* on 2022, and only then score 2023–2025 once.